# Tools 

In [ ]:
# 

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like parakeets, macaws, cockatiels, and others are known for their ability to mimic human speech. But why do they do that? Is it an evolutionary advantage, or is there another reason?\n\nFirst, I remember that parrots have a unique vocal organ called the syrinx. It\'s different from the human larynx. The syrinx is located at the base of the trachea, and it allows them to produce a wide range of sounds. But how does that relate to mimicking human speech? Maybe their syrinx is more flexible, allowing them to vary their sounds more than other birds.\n\nI also know that parrots are highly social animals. They live in flocks in the wild, so communication is crucial for them. Mimicking sounds might help them integrate into the flock, identify other members, or warn of dangers. In captivity, maybe they mimic humans as a way to

In [2]:
# Creating a simple tool to get the weather at a location, and then binding 
# it to the model so that the model can use it in its reasoning process.

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
# Now when we ask the model a question that requires the use of the tool, 
# it will call the tool and include the tool's output in its response.
response = model_with_tools.invoke("What's the weather like in Mascow?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Mascow. First, I need to check if Mascow is a valid location. Wait, Mascow isn't a real city I recognize. Maybe they meant Moscow? That's a common misspelling. Let me confirm. If the user intended Moscow, Russia, then I can proceed. Since the function requires a location parameter, I should use the correct spelling. I'll call the get_weather function with Moscow as the location. That should provide the accurate weather information they need.\n", 'tool_calls': [{'id': 'dyw37dvwt', 'function': {'arguments': '{"location":"Moscow"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 155, 'total_tokens': 287, 'completion_time': 0.248755896, 'completion_tokens_details': {'reasoning_tokens': 107}, 'prompt_time': 0.006473662, 'prompt_tokens_details': None, 'queue_time': 0.054100453, 'total_time': 0.255229558}, 'model_name': 

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Moscow?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Moscow is sunny.


In [9]:
# To check the intermediate steps, 
# you can print the tool calls and their results before passing them back to the model for the final response. 
# This way, you can see exactly how the model is using the tools and what information it is getting from them.
for tool_call in ai_msg.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    tool_result = get_weather.invoke(tool_call)
    print(f"Tool Result: {tool_result.text}")


Tool: get_weather
Args: {'location': 'Moscow'}
Tool Result: It's sunny in Moscow


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Moscow?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Moscow. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Moscow, I need to call this function with "Moscow" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'phfzd97pg', 'function': {'arguments': '{"location":"Moscow"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 153, 'total_tokens': 248, 'completion_time': 0.147825506, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.009790886, 'prompt_tokens_details': None, 'queue_time': 0.162637863, 'total_time': 0.157616392}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_deman